# Learner Notebook — Day 1  
## Real SQLite Memory for an HR Bot

**Session length:** 2.5 hours  
**Audience:** Python developers  
**Project goal:** Build the memory foundation of an HR Bot using real local persistence, not mock memory.

Today you will implement:

1. SQLite-backed employee and leave lookup tools.
2. SQLite-backed episodic memory for recent chat messages.
3. SQLite-backed short-term memory for compressed session summaries.
4. A visible employee-ID flow:
   - user has not provided employee ID → bot asks for it,
   - employee ID already exists in the present chat session → bot uses memory.
5. A compaction rule:
   - keep only the last 10 chat messages in episodic memory,
   - summarize older messages into short-term memory,
   - include both the summary and the last 10 messages in the LLM prompt.

## Colab-first and no-paid-API design

The core lab uses SQLite, deterministic summarisation, and local retrieval so every learner can complete it without purchasing API credits. An optional production extension may use Groq or OpenRouter for final response generation, with keys stored in Colab Secrets.

## Why SQLite Instead of Mock Functions?

SQLite is a real database. It writes to a local `.db` file and persists after the notebook stops.

We use SQLite as the default workshop path because:

- it is built into Python through `sqlite3`;
- students do not need to install a server;
- every learner can inspect the tables;
- the same ideas transfer to Redis, PostgreSQL, or a cloud database later.

Redis is still a strong production option for high-speed episodic memory, but it adds setup friction. In this lab, SQLite gives us a real build without environment blockers.

## Files You Need Beside This Notebook

In Colab, upload these files through the Files panel; locally, keep them beside the notebook:

- `employees.xlsx`
- `leave_balance.xlsx`

This notebook will create a local SQLite file named:

- `hr_bot_memory.db`

## Day 1 Agenda — 2.5 Hours

| Time | Segment | Output |
|---:|---|---|
| 0:00–0:15 | Memory problem setup | Understand why HR Bot needs memory |
| 0:15–0:35 | Memory types | Distinguish episodic, short-term, long-term, and operational data |
| 0:35–1:05 | SQLite setup | Load HR sample data into real SQLite tables |
| 1:05–1:15 | Break | Reset |
| 1:15–1:45 | Employee-ID memory flow | Bot asks for ID vs uses ID from current session |
| 1:45–2:15 | Last-10-message episodic memory | Compact older messages into a short-term summary |
| 2:15–2:30 | Debrief | Explain why short-term memory improves agent behavior |

# 1. Setup
Run the cells below. No database server is required.

In [ ]:
/content/employees.xlsx

In [ ]:
%pip install -q pandas openpyxl

from pathlib import Path
import sqlite3
import pandas as pd
import json
import re
from datetime import datetime


def locate_file(filename: str) -> Path:
    """Find a support file in common local and Google Colab locations."""
    candidates = [Path(filename), Path('/content') / filename]
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidates.append(parent / filename)
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError(
        f"Could not find {filename}. In Colab, upload it through the Files panel or run files.upload()."
    )


EMPLOYEE_FILE = locate_file("employees.xlsx")
LEAVE_FILE = locate_file("leave_balance.xlsx")
DATA_DIR = Path('/content') if Path('/content').exists() else Path.cwd()
DB_FILE = DATA_DIR / "hr_bot_memory.db"

print("Employee file:", EMPLOYEE_FILE)
print("Leave file:", LEAVE_FILE)
print("Using database:", DB_FILE.resolve())

Employee file: /content/employees.xlsx
Leave file: /content/leave_balance.xlsx
Using database: /content/hr_bot_memory.db


In [ ]:
def load_source_files_to_sqlite(db_file=DB_FILE):
    """Load the supplied HR sample spreadsheets into real SQLite tables."""
    employees = pd.read_excel(EMPLOYEE_FILE)

    leave_raw = pd.read_excel(LEAVE_FILE, header=2)
    leave = leave_raw[leave_raw["Employee ID"].astype(str).str.startswith("EMP-", na=False)].copy()

    leave = leave.rename(columns={
        "Entitled": "annual_entitled",
        "Taken": "annual_taken",
        "Balance": "annual_balance",
        "Entitled.1": "sick_entitled",
        "Taken.1": "sick_taken",
        "Balance.1": "sick_balance",
        "Entitled.2": "casual_entitled",
        "Taken.2": "casual_taken",
        "Balance.2": "casual_balance",
        "Comp Off Balance": "comp_off_balance",
        "Leave Year": "leave_year",
        "Status": "status",
    })

    with sqlite3.connect(db_file) as conn:
        employees.to_sql("employees", conn, if_exists="replace", index=False)
        leave.to_sql("leave_balances", conn, if_exists="replace", index=False)

    return employees.shape, leave.shape

employee_shape, leave_shape = load_source_files_to_sqlite()
employee_shape, leave_shape

((20, 9), (20, 14))

# 2. Inspect the SQLite Database

The two operational data sources are now SQLite tables:

- `employees`
- `leave_balances`

These are not LLM memory. They are factual tools the bot can query.

In [ ]:
with sqlite3.connect(DB_FILE) as conn:
    table_names = pd.read_sql(
        "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name",
        conn
    )

table_names

,name
0,employees
1,leave_balances


In [ ]:
def query_one(sql: str, params=()):
    """Return one SQLite row as a Python dict."""
    with sqlite3.connect(DB_FILE) as conn:
        conn.row_factory = sqlite3.Row
        row = conn.execute(sql, params).fetchone()
    return dict(row) if row else None

def get_employee_details(emp_id: str) -> dict:
    row = query_one(
        """
        SELECT "Employee ID" AS employee_id,
               Name AS name,
               Designation AS designation,
               "Reporting Manager" AS reporting_manager
        FROM employees
        WHERE "Employee ID" = ?
        """,
        (emp_id,),
    )
    return row or {"error": f"Employee {emp_id} not found"}

def get_leave_balance(emp_id: str) -> dict:
    row = query_one(
        """
        SELECT "Employee ID" AS employee_id,
               Name AS name,
               annual_balance,
               sick_balance,
               casual_balance,
               comp_off_balance,
               leave_year,
               status
        FROM leave_balances
        WHERE "Employee ID" = ?
        """,
        (emp_id,),
    )
    return row or {"error": f"No leave record for {emp_id}"}

print(get_employee_details("EMP-1001"))
print(get_leave_balance("EMP-1001"))

{'employee_id': 'EMP-1001', 'name': 'Priya Sharma', 'designation': 'Software Engineer', 'reporting_manager': 'Ravi Kumar'}
{'employee_id': 'EMP-1001', 'name': 'Priya Sharma', 'annual_balance': 13, 'sick_balance': 9, 'casual_balance': 4, 'comp_off_balance': 3, 'leave_year': '2025-26', 'status': '✔ Healthy'}


## Checkpoint

Answer in one sentence:

1. Why should an HR Bot look up leave balance from a database instead of asking the LLM to guess?
2. Which information should be treated as operational data rather than conversation memory?

# 3. Create Real Memory Tables

We will use two tables:

| Table | Memory type | What it stores |
|---|---|---|
| `chat_messages` | Episodic memory | Only the last 10 messages in the present chat session |
| `session_memory` | Short-term memory | Compressed summary and the employee ID for the present chat session |

The important rule is:

> Episodic memory is small and recent. Short-term memory is compact and durable.

In [ ]:
MAX_EPISODIC_MESSAGES = 10

def init_memory_tables(db_file=DB_FILE):
    """Create real persistent memory tables in SQLite."""
    with sqlite3.connect(db_file) as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS chat_messages (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                session_id TEXT NOT NULL,
                role TEXT NOT NULL CHECK(role IN ('user', 'assistant', 'system')),
                content TEXT NOT NULL,
                created_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP
            )
        """)
        conn.execute("""
            CREATE TABLE IF NOT EXISTS session_memory (
                session_id TEXT PRIMARY KEY,
                summary TEXT NOT NULL DEFAULT '',
                current_employee_id TEXT,
                updated_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP
            )
        """)
        conn.execute("CREATE INDEX IF NOT EXISTS idx_chat_session_id ON chat_messages(session_id, id)")
        conn.commit()

init_memory_tables()
print("Memory tables ready.")

Memory tables ready.


In [ ]:
def extract_emp_id(text: str):
    """Extract employee IDs such as EMP-1001 from free text."""
    match = re.search(r"\bEMP-\d+\b", text.upper())
    return match.group(0) if match else None

def get_recent_messages(session_id: str, limit: int = MAX_EPISODIC_MESSAGES):
    """Fetch only the recent episodic memory window."""
    with sqlite3.connect(DB_FILE) as conn:
        conn.row_factory = sqlite3.Row
        rows = conn.execute(
            """
            SELECT id, role, content, created_at
            FROM chat_messages
            WHERE session_id = ?
            ORDER BY id DESC
            LIMIT ?
            """,
            (session_id, limit),
        ).fetchall()
    return [dict(row) for row in reversed(rows)]

def get_session_memory(session_id: str):
    with sqlite3.connect(DB_FILE) as conn:
        conn.row_factory = sqlite3.Row
        row = conn.execute(
            "SELECT session_id, summary, current_employee_id, updated_at FROM session_memory WHERE session_id = ?",
            (session_id,),
        ).fetchone()
    if row:
        return dict(row)
    return {"session_id": session_id, "summary": "", "current_employee_id": None, "updated_at": None}

def upsert_session_memory(session_id: str, summary=None, current_employee_id=None):
    """Update durable short-term memory. Existing values are preserved unless a new value is supplied."""
    existing = get_session_memory(session_id)
    final_summary = existing["summary"] if summary is None else summary
    final_emp_id = existing["current_employee_id"] if current_employee_id is None else current_employee_id

    with sqlite3.connect(DB_FILE) as conn:
        conn.execute(
            """
            INSERT INTO session_memory(session_id, summary, current_employee_id, updated_at)
            VALUES (?, ?, ?, CURRENT_TIMESTAMP)
            ON CONFLICT(session_id) DO UPDATE SET
                summary = excluded.summary,
                current_employee_id = excluded.current_employee_id,
                updated_at = CURRENT_TIMESTAMP
            """,
            (session_id, final_summary, final_emp_id),
        )
        conn.commit()

def compress_messages(existing_summary: str, older_messages: list[dict]) -> str:
    """
    Compress older chat messages into durable short-term memory.

    In production, this can call an LLM summarizer. For the workshop, this is a deterministic
    compression step so every learner can run it without an API key.
    """
    if not older_messages:
        return existing_summary or ""

    employee_ids = sorted({
        emp_id
        for msg in older_messages
        for emp_id in [extract_emp_id(msg["content"])]
        if emp_id
    })

    topic_keywords = []
    text = " ".join(msg["content"].lower() for msg in older_messages)
    for keyword in ["leave", "manager", "policy", "sick", "casual", "annual", "approval"]:
        if keyword in text:
            topic_keywords.append(keyword)

    compressed_turns = " | ".join(
        f'{msg["role"]}: {msg["content"][:90].replace(chr(10), " ")}'
        for msg in older_messages[-6:]
    )

    new_summary_piece = (
        f"Compressed {len(older_messages)} older messages. "
        f"Employee IDs mentioned: {employee_ids or 'none'}. "
        f"Topics: {topic_keywords or 'general HR chat'}. "
        f"Recent older context: {compressed_turns}"
    )

    if existing_summary:
        combined = existing_summary.strip() + "\n" + new_summary_piece
    else:
        combined = new_summary_piece

    # Keep the durable summary compact enough to place into prompts.
    return combined[-1800:]

def compact_episodic_memory(session_id: str, keep_last: int = MAX_EPISODIC_MESSAGES):
    """
    Keep only the last N chat messages in episodic memory.
    Move everything before that into short-term summary memory.
    """
    with sqlite3.connect(DB_FILE) as conn:
        conn.row_factory = sqlite3.Row
        rows = conn.execute(
            """
            SELECT id, role, content, created_at
            FROM chat_messages
            WHERE session_id = ?
            ORDER BY id ASC
            """,
            (session_id,),
        ).fetchall()

    rows = [dict(row) for row in rows]
    if len(rows) <= keep_last:
        return {"compacted": 0, "kept": len(rows)}

    older = rows[:-keep_last]
    keep = rows[-keep_last:]

    existing = get_session_memory(session_id)
    updated_summary = compress_messages(existing["summary"], older)
    upsert_session_memory(session_id, summary=updated_summary)

    older_ids = [row["id"] for row in older]
    placeholders = ",".join("?" for _ in older_ids)
    with sqlite3.connect(DB_FILE) as conn:
        conn.execute(f"DELETE FROM chat_messages WHERE id IN ({placeholders})", older_ids)
        conn.commit()

    return {"compacted": len(older), "kept": len(keep)}

def save_message(session_id: str, role: str, content: str):
    """Save a chat message, then enforce the 'last 10 messages only' episodic window."""
    with sqlite3.connect(DB_FILE) as conn:
        conn.execute(
            "INSERT INTO chat_messages(session_id, role, content) VALUES (?, ?, ?)",
            (session_id, role, content),
        )
        conn.commit()
    return compact_episodic_memory(session_id)

def reset_session(session_id: str):
    """Convenience function for repeated demos."""
    with sqlite3.connect(DB_FILE) as conn:
        conn.execute("DELETE FROM chat_messages WHERE session_id = ?", (session_id,))
        conn.execute("DELETE FROM session_memory WHERE session_id = ?", (session_id,))
        conn.commit()

print("Memory functions ready.")

Memory functions ready.


# 4. Build the Employee-ID Flow

The bot must behave differently in two cases:

## Case A — Employee ID missing in the present chat session
User asks:  
> “How many leave days do I have?”

The bot should not guess. It should ask for the employee ID.

## Case B — Employee ID already provided in this present chat session
User says:  
> “My employee ID is EMP-1001.”

Later the user asks:  
> “How many leave days do I have?”

The bot should use the employee ID from memory.

In [ ]:
def resolve_employee_id(session_id: str, user_message: str):
    """
    Resolve employee ID in this order:
    1. Current user message.
    2. Durable short-term memory for the present chat session.
    3. Last 10 episodic chat messages.
    4. Missing → bot should ask the user.
    """
    emp_id = extract_emp_id(user_message)
    if emp_id:
        upsert_session_memory(session_id, current_employee_id=emp_id)
        return emp_id, "current_message"

    session_memory = get_session_memory(session_id)
    if session_memory["current_employee_id"]:
        return session_memory["current_employee_id"], "short_term_session_memory"

    for msg in reversed(get_recent_messages(session_id)):
        emp_id = extract_emp_id(msg["content"])
        if emp_id:
            upsert_session_memory(session_id, current_employee_id=emp_id)
            return emp_id, "episodic_recent_messages"

    return None, "missing"

def answer_leave_balance(emp_id: str) -> str:
    details = get_leave_balance(emp_id)
    if "error" in details:
        return details["error"]
    return (
        f'{details["name"]} ({emp_id}) has '
        f'{int(details["annual_balance"])} annual leave days, '
        f'{int(details["sick_balance"])} sick leave days, '
        f'{int(details["casual_balance"])} casual leave days, and '
        f'{int(details["comp_off_balance"])} comp-off days available '
        f'for {details["leave_year"]}.'
    )

def answer_manager(emp_id: str) -> str:
    details = get_employee_details(emp_id)
    if "error" in details:
        return details["error"]
    return f'{details["name"]} ({emp_id}) reports to {details["reporting_manager"]}.'

def hr_bot(session_id: str, user_message: str) -> str:
    """
    A small but real tool-orchestration bot:
    - persists every turn to SQLite,
    - resolves employee ID from current/session/recent memory,
    - calls SQLite tools for facts,
    - asks for missing employee ID when needed.
    """
    save_message(session_id, "user", user_message)

    emp_id, source = resolve_employee_id(session_id, user_message)
    lower = user_message.lower()

    needs_employee_record = any(word in lower for word in ["my", "leave", "manager", "balance", "days"])

    if not emp_id and needs_employee_record:
        response = (
            "I can help with that, but I do not know which employee record to check in this chat session. "
            "Please share your employee ID, for example: EMP-1001."
        )
    elif "leave" in lower or "balance" in lower or "days" in lower:
        response = answer_leave_balance(emp_id) + f" [Employee ID source: {source}]"
    elif "manager" in lower or "reporting" in lower:
        response = answer_manager(emp_id) + f" [Employee ID source: {source}]"
    elif emp_id:
        response = f"Thanks. I will use {emp_id} for this chat session. [Employee ID source: {source}]"
    else:
        response = "I can help with leave balance, reporting manager, and HR policy questions."

    save_message(session_id, "assistant", response)
    return response

## Demo A — User Has Not Provided Employee ID

Expected behavior: the HR Bot asks for the employee ID.

In [ ]:
session_id = "demo_no_employee_id"
reset_session(session_id)

print(hr_bot(session_id, "How many annual leave days do I have?"))

I can help with that, but I do not know which employee record to check in this chat session. Please share your employee ID, for example: EMP-1001.


## Demo B — Employee ID Was Already Provided in the Present Chat Session

Expected behavior:

1. User provides employee ID once.
2. Bot stores it in short-term session memory.
3. Next question does not include employee ID.
4. Bot still answers correctly.

In [ ]:
session_id = "demo_employee_id_already_known"
reset_session(session_id)

print(hr_bot(session_id, "My employee ID is EMP-1001."))
print(hr_bot(session_id, "How many annual leave days do I have?"))
print(hr_bot(session_id, "Who is my reporting manager?"))

get_session_memory(session_id)

Thanks. I will use EMP-1001 for this chat session. [Employee ID source: current_message]
Priya Sharma (EMP-1001) has 13 annual leave days, 9 sick leave days, 4 casual leave days, and 3 comp-off days available for 2025-26. [Employee ID source: short_term_session_memory]
Priya Sharma (EMP-1001) reports to Ravi Kumar. [Employee ID source: short_term_session_memory]


{'session_id': 'demo_employee_id_already_known',
 'summary': '',
 'current_employee_id': 'EMP-1001',
 'updated_at': '2026-07-13 16:08:10'}

## Inspect the Actual Messages Saved

This is the recent episodic memory window for the current session.

In [ ]:
pd.DataFrame(get_recent_messages(session_id))

,id,role,content,created_at
0,3,user,My employee ID is EMP-1001.,2026-07-13 16:08:10
1,4,assistant,Thanks. I will use EMP-1001 for this chat sess...,2026-07-13 16:08:10
2,5,user,How many annual leave days do I have?,2026-07-13 16:08:10
3,6,assistant,Priya Sharma (EMP-1001) has 13 annual leave da...,2026-07-13 16:08:10
4,7,user,Who is my reporting manager?,2026-07-13 16:08:10
5,8,assistant,Priya Sharma (EMP-1001) reports to Ravi Kumar....,2026-07-13 16:08:10


# 5. Keep Only the Last 10 Chat Messages

Now we will force the conversation to exceed 10 messages.

Rule:

- `chat_messages` keeps only the last 10 messages.
- older messages are compressed into `session_memory.summary`.

This is what prevents the LLM prompt from becoming too large while still preserving useful context.

In [ ]:
session_id = "demo_last_10_compaction"
reset_session(session_id)

for i in range(1, 9):  # 8 turns = 16 messages
    save_message(session_id, "user", f"Turn {i}: I am asking about annual leave, manager approval, and EMP-1001.")
    save_message(session_id, "assistant", f"Turn {i}: I answered using the HR tools.")

recent = get_recent_messages(session_id)
memory = get_session_memory(session_id)

print("Messages kept in episodic memory:", len(recent))
print("Current employee ID:", memory["current_employee_id"])
print("\nCompressed short-term summary:\n")
print(memory["summary"])

pd.DataFrame(recent)

Messages kept in episodic memory: 10
Current employee ID: None

Compressed short-term summary:

Compressed 1 older messages. Employee IDs mentioned: ['EMP-1001']. Topics: ['leave', 'manager', 'annual', 'approval']. Recent older context: user: Turn 1: I am asking about annual leave, manager approval, and EMP-1001.
Compressed 1 older messages. Employee IDs mentioned: none. Topics: general HR chat. Recent older context: assistant: Turn 1: I answered using the HR tools.
Compressed 1 older messages. Employee IDs mentioned: ['EMP-1001']. Topics: ['leave', 'manager', 'annual', 'approval']. Recent older context: user: Turn 2: I am asking about annual leave, manager approval, and EMP-1001.
Compressed 1 older messages. Employee IDs mentioned: none. Topics: general HR chat. Recent older context: assistant: Turn 2: I answered using the HR tools.
Compressed 1 older messages. Employee IDs mentioned: ['EMP-1001']. Topics: ['leave', 'manager', 'annual', 'approval']. Recent older context: user: Turn 3:

,id,role,content,created_at
0,15,user,"Turn 4: I am asking about annual leave, manage...",2026-07-13 16:08:10
1,16,assistant,Turn 4: I answered using the HR tools.,2026-07-13 16:08:10
2,17,user,"Turn 5: I am asking about annual leave, manage...",2026-07-13 16:08:10
3,18,assistant,Turn 5: I answered using the HR tools.,2026-07-13 16:08:10
4,19,user,"Turn 6: I am asking about annual leave, manage...",2026-07-13 16:08:10
5,20,assistant,Turn 6: I answered using the HR tools.,2026-07-13 16:08:10
6,21,user,"Turn 7: I am asking about annual leave, manage...",2026-07-13 16:08:10
7,22,assistant,Turn 7: I answered using the HR tools.,2026-07-13 16:08:10
8,23,user,"Turn 8: I am asking about annual leave, manage...",2026-07-13 16:08:10
9,24,assistant,Turn 8: I answered using the HR tools.,2026-07-13 16:08:10


## What Just Happened?

The earlier messages did not disappear entirely.

They were compressed into short-term session memory, while the newest 10 messages remain available as episodic memory.

This models a common agent pattern:

```text
Older chat history → compress/summarize → short-term memory
Last 10 chat messages → keep verbatim → episodic memory
Both pieces → included in the LLM prompt
```

# 6. Build the Prompt Package

An LLM should not receive unlimited chat history. Instead, it receives:

1. the short-term session summary,
2. the last 10 episodic messages,
3. the current user message,
4. retrieved policy snippets or tool outputs when needed.

Run the next cell to see the exact prompt bundle.

In [ ]:
def build_prompt_bundle(session_id: str, current_user_message: str) -> str:
    """
    This shows exactly what would be sent to the LLM:
    - compressed short-term session summary,
    - only the last 10 episodic chat messages,
    - the current user message.
    """
    memory = get_session_memory(session_id)
    recent = get_recent_messages(session_id, limit=MAX_EPISODIC_MESSAGES)

    recent_text = "\n".join(
        f'{m["role"].upper()}: {m["content"]}'
        for m in recent
    ) or "(no recent messages)"

    return f"""
SYSTEM:
You are a careful HR assistant. Use only the provided memory and tools. Do not invent employee data.

SHORT-TERM SESSION SUMMARY:
{memory["summary"] or "(no compressed summary yet)"}

CURRENT EMPLOYEE ID STORED FOR THIS SESSION:
{memory["current_employee_id"] or "(not provided yet)"}

EPISODIC MEMORY — LAST {MAX_EPISODIC_MESSAGES} CHAT MESSAGES ONLY:
{recent_text}

CURRENT USER MESSAGE:
{current_user_message}
""".strip()

In [ ]:
print(build_prompt_bundle(
    "demo_last_10_compaction",
    "Given our earlier discussion, can I take 3 annual leave days next week?"
))

SYSTEM:
You are a careful HR assistant. Use only the provided memory and tools. Do not invent employee data.

SHORT-TERM SESSION SUMMARY:
Compressed 1 older messages. Employee IDs mentioned: ['EMP-1001']. Topics: ['leave', 'manager', 'annual', 'approval']. Recent older context: user: Turn 1: I am asking about annual leave, manager approval, and EMP-1001.
Compressed 1 older messages. Employee IDs mentioned: none. Topics: general HR chat. Recent older context: assistant: Turn 1: I answered using the HR tools.
Compressed 1 older messages. Employee IDs mentioned: ['EMP-1001']. Topics: ['leave', 'manager', 'annual', 'approval']. Recent older context: user: Turn 2: I am asking about annual leave, manager approval, and EMP-1001.
Compressed 1 older messages. Employee IDs mentioned: none. Topics: general HR chat. Recent older context: assistant: Turn 2: I answered using the HR tools.
Compressed 1 older messages. Employee IDs mentioned: ['EMP-1001']. Topics: ['leave', 'manager', 'annual', 'appro

## Practice Task

Modify the demo:

1. Change the session ID to your own name or initials.
2. First ask a leave question without giving an employee ID.
3. Then provide an employee ID.
4. Ask the same leave question again.
5. Add enough messages to trigger compaction.
6. Inspect `chat_messages` and `session_memory`.

Use this query to inspect raw tables:

In [ ]:
from IPython.display import display

with sqlite3.connect(DB_FILE) as conn:
    display(pd.read_sql("SELECT * FROM session_memory", conn))
    display(pd.read_sql("SELECT * FROM chat_messages ORDER BY session_id, id", conn).tail(20))

,session_id,summary,current_employee_id,updated_at
0,demo_employee_id_already_known,,EMP-1001,2026-07-13 16:08:10
1,demo_last_10_compaction,Compressed 1 older messages. Employee IDs ment...,None,2026-07-13 16:08:10


,id,session_id,role,content,created_at
0,3,demo_employee_id_already_known,user,My employee ID is EMP-1001.,2026-07-13 16:08:10
1,4,demo_employee_id_already_known,assistant,Thanks. I will use EMP-1001 for this chat sess...,2026-07-13 16:08:10
2,5,demo_employee_id_already_known,user,How many annual leave days do I have?,2026-07-13 16:08:10
3,6,demo_employee_id_already_known,assistant,Priya Sharma (EMP-1001) has 13 annual leave da...,2026-07-13 16:08:10
4,7,demo_employee_id_already_known,user,Who is my reporting manager?,2026-07-13 16:08:10
5,8,demo_employee_id_already_known,assistant,Priya Sharma (EMP-1001) reports to Ravi Kumar....,2026-07-13 16:08:10
6,15,demo_last_10_compaction,user,"Turn 4: I am asking about annual leave, manage...",2026-07-13 16:08:10
7,16,demo_last_10_compaction,assistant,Turn 4: I answered using the HR tools.,2026-07-13 16:08:10
8,17,demo_last_10_compaction,user,"Turn 5: I am asking about annual leave, manage...",2026-07-13 16:08:10
9,18,demo_last_10_compaction,assistant,Turn 5: I answered using the HR tools.,2026-07-13 16:08:10


# Day 1 Wrap-Up

You built a real memory layer using SQLite.

You should now be able to explain:

- why employee ID should be remembered within a present chat session;
- why the bot must ask for employee ID when it is missing;
- why only the last 10 messages should remain in episodic memory;
- how older messages can be summarized into short-term memory;
- what information belongs in operational tools rather than LLM memory.

## Optional free-provider upgrade

Reuse the provider helper from Week 5 to send the assembled HR prompt to Groq or OpenRouter. Keep employee records and sensitive HR data anonymised in classroom exercises, and do not send real personal data to any hosted model without organisational approval.